In [1]:
import pandas as pd
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = [
    "age", "workclass", "fnlwgt", "education",
    "education_num", "marital_status", "occupation",
    "relationship", "race", "sex", "capital_gain",
    "capital_loss", "hours_per_week", "native_country",
    "income"]
df = pd.read_csv(url,names=columns,skipinitialspace=True,na_values="?")
print(df.head())
print(df.shape)
print(df.info())
print(df.isnull().sum())
print(df["income"].value_counts())

   age         workclass  fnlwgt  education  education_num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   
3   53           Private  234721       11th              7   
4   28           Private  338409  Bachelors             13   

       marital_status         occupation   relationship   race     sex  \
0       Never-married       Adm-clerical  Not-in-family  White    Male   
1  Married-civ-spouse    Exec-managerial        Husband  White    Male   
2            Divorced  Handlers-cleaners  Not-in-family  White    Male   
3  Married-civ-spouse  Handlers-cleaners        Husband  Black    Male   
4  Married-civ-spouse     Prof-specialty           Wife  Black  Female   

   capital_gain  capital_loss  hours_per_week native_country income  
0          2174             0              40  United-States  <=50K  
1             0             0             

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
X = df.drop("income", axis=1)
y = df["income"].str.strip().map({"<=50K": 0, ">50K": 1})
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "str", "category"]).columns
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)
numeric_pipeline=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("num", numeric_pipeline, numeric_features),("cat", categorical_pipeline, categorical_features)])
model_pipeline = Pipeline([("preprocessor", preprocessor),("model", RandomForestClassifier(random_state=42,class_weight="balanced"))])
model_pipeline.fit(X_train, y_train)
print("Pipeline trained successfully!")

Pipeline trained successfully!


In [3]:
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced"
    ),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced", random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42)}
results = []
for name, classifier in models.items():
    pipeline = Pipeline([("preprocessor", preprocessor),("model", classifier)])
    scores = cross_val_score(pipeline,X,y,cv=cv,scoring="f1",n_jobs=-1)
    results.append({"Model": name,"Mean F1": scores.mean(),"Std F1": scores.std()})
results_df = pd.DataFrame(results)
print(results_df.sort_values("Mean F1", ascending=False))

                 Model   Mean F1    Std F1
0  Logistic Regression  0.719215  0.005558
2        Random Forest  0.694426  0.004970
1        Decision Tree  0.676108  0.012013
3    Gradient Boosting  0.645839  0.005867


In [4]:
import joblib
model_pipeline.fit(X_train, y_train)
joblib.dump(model_pipeline, "adult_income_pipeline.pkl")
print("Pipeline saved successfully!")

Pipeline saved successfully!
